<a href="https://colab.research.google.com/github/Yahimeen/IA_AplicadaLLama_Challenge_02_03/blob/main/Hands_On_Integraci%C3%B3n_entre_Llama_y_WhatsApp_Challenge_02_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: INTEGRACIÓN ENTRE LLAMA Y WHATSAPP**

Una vez vista la masterclass ***Integración entre Llama y WhatsApp***, se proporciona el siguiente ***colab*** para que **el alumnado pueda practicar** haciendo uso del lenguaje de programación ***python*** y el modelo de ***llama 3.1*** de Meta.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1ks58AJ1dS_aeNw_Q4pnYR05Dokr0Dwi6?usp=drive_link).

## **LÓGICA DE NEGOCIO Y MODELO**

### **CONFIGURACIÓN DEL ENTORNO**

#### **COLAB SECRETS**

Para no exponer tu API key directamente en el código, Colab ofrece un panel de Secrets (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la API key para guardarla dentro de una variable y usarla dentro del notebook.

In [ ]:
# Leer credenciales desde Colab Secrets


#### **INSTALAR Y ARRANCAR OLLAMA EN COLAB**

Este módulo corre en Colab, así que Ollama no viene preinstalado en la máquina virtual: se instala y se arranca en segundo plano al inicio de cada sesión.

In [ ]:
# Instalar Ollama (requiere zstd, que Colab no trae por defecto)


In [ ]:
# Iniciar el servidor de Ollama en segundo plano


#### **INSTALAR Y LEVANTAR LLAMA STACK**

In [ ]:
# Instalar dependencias


In [ ]:
# Instalar Llama Stack (servidor + cliente) y levantarlo con Ollama registrado


### **CONSTRUCCIÓN DE LA LÓGICA DEL ASISTENTE**

In [ ]:
# Configurar el servidor del webhook


In [ ]:
# Definir función de normalización de número


#### **CONSTRUIR LA MEMORIA DE CONVERSACIÓN**

Se define una estructura en memoria que asocia cada número de teléfono con su `conversation_id` de **Llama Stack** — sin esto, Llama trataría cada mensaje como si fuera el primero de la conversación.

In [ ]:
# Una conversacion de Llama Stack por numero de telefono


#### **DEFINIR LA HERRAMIENTA: CONSULTAR DISPONIBILIDAD Y PREGUNTAS FRECUENTES**

Hasta el Tema anterior, Llama solo generaba texto a partir de texto — si un usuario preguntaba por un horario, el modelo podía inventar una respuesta que sonara razonable, sin ninguna garantía de que fuera real. **Function calling** resuelve esto: se le describe a Llama una función real de tu código (nombre, qué hace, qué parámetros necesita), y el modelo, cuando lo considera necesario, pide que se ejecute esa función en vez de responder directamente.

Aquí se definen dos herramientas: `consultar_disponibilidad`, que revisa horarios reales del taller, y `buscar_pregunta_frecuente`, que busca la respuesta a dudas comunes (horarios, garantía, formas de pago) en una base de preguntas frecuentes real — una alternativa más ligera a montar un pipeline de RAG completo (embeddings + recuperación) aparte: en vez de eso, la búsqueda se expone como una función más que Llama puede pedir ejecutar, bajo el mismo mecanismo de function calling.

In [ ]:
# Definir la funcion real y su descripcion para Llama: consultar disponibilidad


In [ ]:
# Definir la funcion real y su descripcion para Llama: buscar pregunta frecuente


#### **GENERAR LA RESPUESTA CON HISTORIAL Y FUNCTION CALLING**

Se reenvía el historial reciente a Llama junto con las dos herramientas disponibles. Si el modelo pide ejecutar alguna (`respuesta.output` trae uno o más elementos con `type == "function_call"`), el código la ejecuta de verdad con los argumentos que Llama propuso, le devuelve el resultado en una segunda llamada, y solo entonces se genera la respuesta final en lenguaje natural. Las reglas fijas (urgencia y escalamiento) se resuelven antes de llegar aquí — ver la sección de endpoints.

In [ ]:
# Definir la funcion de generacion de respuesta con memoria y function calling


## **DESPLIEGUE DEL SERVIDOR**

### **DEFINIR LOS ENDPOINTS DEL WEBHOOK**

Se definen los dos endpoints que WhatsApp necesita: uno para la verificación inicial del webhook, y otro para recibir cada mensaje entrante. La recepción delega el trabajo pesado a `BackgroundTasks`, para responderle a Meta de inmediato sin esperar a que Llama genere la respuesta — evitando así que Meta reintente la entrega y se dupliquen las respuestas.

In [ ]:
# Definir endpoint de verificacion de credenciales


In [ ]:
# Definir endpoint de recepcion de mensajes


In [ ]:
# Definir funcion de envio de respuesta


In [ ]:
# Un lock por numero de telefono, para serializar mensajes del mismo usuario


In [ ]:
# Definir funcion de procesamiento: reglas fijas de urgencia, escalamiento y saludo, o respuesta de Llama


### **EXPONER EL SERVIDOR CON NGROK**

Se abre un túnel público con ngrok y se levanta el servidor en un hilo aparte, para que Meta pueda entregarle los mensajes reales al webhook. La URL impresa (+ `/webhook`) es la que se configura como *Callback URL* en Meta for Developers (WhatsApp > Configuración de la API > Configurar webhooks).

In [ ]:
# Instalar e iniciar tunel publico con ngrok


### **PROBAR EL FLUJO COMPLETO**

Se manda un mensaje preguntando por horarios disponibles en una fecha específica, para confirmar que la respuesta viene de `consultar_disponibilidad` y no de una invención de Llama; y un segundo mensaje pidiendo hablar con una persona, para confirmar que la regla fija de escalamiento sigue funcionando.

In [ ]:
# Escenario 1: Mandar mensajes al bot por disponibilidad general

# Mensaje 1:
# Buenas tardes.

# Mensaje 2:
# Necesito llevar mi carro al taller, ¿qué días tienen espacio?

# Mensaje 3:
# Perfecto, ¿y a qué hora el 6 de octubre?

In [ ]:
# Escenario 2: Mandar mensajes al bot por tema de urgencia

# Mensaje 1:
# Hola. Necesito ayuda.
# Acabo de chocar y mi carro no arranca.

# Mensaje 2:
# Comunícame con un asistente humano.

In [ ]:
# Escenario 3: Mandar mensajes al bot por pago y cotización

# Mensaje 1:
# Buenas, ¿cuánto cuesta una cotización?

# Mensaje 2:
# ¿Y qué métodos de pago aceptan?

# Mensaje 3:
# Perfecto, entonces agéndame para el 7 de octubre a las 10.

In [ ]:
# Escenario 4: Mandar mensajes al bot por temas fuera de alcance

# Mensaje 1:
# Hola, ¿tienen servicio de grúa?

# Mensaje 2:
# ¿También atienden motos, o solo carros?

In [ ]:
# Visualizar resultados


In [ ]:
# Verificar almacenamiento en Llama Stack


### **DIAGNÓSTICO Y REINICIO**

Si el bot responde con datos que no corresponden al taller (fechas, textos u otro negocio), antes de reiniciar todo el entorno revisa primero si simplemente falta volver a correr alguna celda de arriba.

In [ ]:
# Reiniciar el almacenamiento en Llama Stack


In [ ]:
# Limpiar los registros en memoria, para una prueba nueva sin ruido de mensajes anteriores


In [ ]:
# Finalizar proceso de tunelizacion


## **CHALLENGE: AGENTE DE ATENCIÓN A PEDIDOS**

Una vez visto el ***Hands-On: Integración Llama + WhatsApp***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un agente similar, pero para la atención a pedidos de una tienda en línea: la misma memoria de conversación en Llama Stack, las mismas reglas fijas (urgencia, escalamiento y saludo) y las mismas dos herramientas de function calling, ahora como `consultar_pedido` y `buscar_pregunta_frecuente`.

**IMPORTANTE:** Para su revisión, es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.

### **INSTRUCCIONES:**

**1. Prepara el entorno:** Instala y arranca Ollama en Colab, descarga el modelo Llama 3.1 8B, instala y levanta **Llama Stack** con Ollama registrado, y carga las credenciales de WhatsApp desde **Colab Secrets**.

In [ ]:
# Instalar Ollama (requiere zstd, que Colab no trae por defecto)


In [ ]:
# Iniciar el servidor de Ollama en segundo plano


In [ ]:
# Instalar dependencias


In [ ]:
# Instalar Llama Stack (servidor + cliente) y levantarlo con Ollama registrado


In [ ]:
# Leer credenciales desde Colab Secrets


**2. Construye la memoria de conversación y las herramientas de function calling:**

   * Igual que en el Hands-On, crea una conversación de **Llama Stack** por número de teléfono (`client.conversations.create()`).
   * Define `consultar_pedido(numero_pedido)`, una función real que consulta el estado de un pedido, junto con su descripción para que Llama la pueda pedir como herramienta. Si el usuario no da un número de pedido, la función debe pedírselo en vez de inventar uno.
   * Define `buscar_pregunta_frecuente(pregunta)`, que busca la respuesta a dudas comunes (envíos, devoluciones, formas de pago, facturas) en una base de preguntas frecuentes real.

In [ ]:
# Conversacion de Llama Stack por numero de telefono


In [ ]:
# Definir la funcion real y su descripcion para Llama: consultar pedido


In [ ]:
# Definir la funcion real y su descripcion para Llama: buscar pregunta frecuente


**3. Genera la respuesta con historial y function calling:** reenvía el historial a Llama junto con las dos herramientas; si el modelo pide ejecutar alguna, ejecútala de verdad y devuelve el resultado antes de generar la respuesta final.

In [ ]:
# Definir la funcion de generacion de respuesta con memoria y function calling


**4. Agrega las reglas fijas y protege el procesamiento por número:**

   * Si el usuario menciona un problema con un cargo (ej. "me cobraron dos veces", "cargo no reconocido", "fraude"), se responde con un mensaje fijo de prioridad, sin pasar por Llama.
   * Si el usuario pide hablar con una persona, se responde con un mensaje fijo de escalamiento.
   * Si el mensaje es solo un saludo ("hola", "buenas tardes"), se responde con un saludo fijo presentando al asistente.
   * Igual que en el Hands-On, usa un lock por número de teléfono para que dos mensajes seguidos del mismo usuario no disparen dos llamadas en paralelo sobre la misma conversación.
   * Registra en `errores_procesamiento` la causa de cada falla (Llama Stack, Ollama o WhatsApp) y avisa al usuario con un mensaje genérico, para que el bot nunca se quede mudo.

In [ ]:
# Un lock por numero de telefono, para serializar mensajes del mismo usuario


In [ ]:
# Definir funcion de procesamiento: reglas fijas de urgencia, escalamiento y saludo, o respuesta de Llama


**5. Define los endpoints del webhook:** usa `BackgroundTasks` para responder de inmediato a Meta, delegando el procesamiento real a la función anterior. Ignora los mensajes que Meta reenvíe con el mismo `wamid`, para no responderlos dos veces.

In [ ]:
# Configurar el servidor del webhook


In [ ]:
# Definir función de normalización de número


In [ ]:
# Definir endpoint de verificacion de credenciales


In [ ]:
# Definir endpoint de recepcion de mensajes


In [ ]:
# Definir funcion de envio de respuesta


**6. Expón el servidor con ngrok y prueba el flujo completo:**

   * Manda un mensaje preguntando por el estado de un pedido y confirma que la respuesta viene de `consultar_pedido` (dato real), no de una invención de Llama.
   * Manda una pregunta frecuente y confirma que dispara `buscar_pregunta_frecuente`.
   * Manda un mensaje sobre un cargo duplicado y confirma que dispara la regla de urgencia, sin pasar por Llama.
   * Manda un mensaje pidiendo hablar con una persona y confirma que dispara la regla de escalamiento.

In [ ]:
# Instalar e iniciar tunel publico con ngrok


In [ ]:
# Escenario 1: Mandar mensajes al bot por tema de pedidos y preguntas frecuentes


In [ ]:
# Escenario 2: Mandar mensajes al bot por tema de cargos


In [ ]:
# Escenario 3: Mandar mensajes al bot sobre un pedido sin dar el numero, y con un numero que no existe


In [ ]:
# Visualizar resultados


In [ ]:
# Verificar almacenamiento en Llama Stack


In [ ]:
# Finalizar proceso de tunelizacion
